In [7]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 

In [8]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df =  df.drop("customerID", axis = 1)

In [9]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


1️⃣ Define Churn Clearly

Churn = Customer who has stopped using the service

In this dataset:
    
- Target variable: Churn

    - Yes → customer has churned

    - No → customer is retained

📌 Operational definition (important):
A customer is considered churned if they **cancel the service within the observation window.**

This definition matters because:

- It fixes label ambiguity

- It aligns model predictions with actionable timelines

Why is Churn Prediction Important?

- **Cost Efficiency**: Retaining existing customers is significantly less expensive than acquiring new ones
- **Targeted Retention**: Enables focused retention efforts on high-risk customers
- **Revenue Protection**: Helps maintain market position and profitability


2️⃣ Business Goal (Real Objective)

🎯 Catch churners early enough to intervene

Not:

- “Maximize accuracy”

But:

- Identify customers likely to churn BEFORE they actually leave

- Enable:

    - retention offers

    - proactive customer support

    - targeted discounts

This immediately tells us:
- False Negatives (missed churners) are very costly

- False Positives are annoying, but survivable

**Problem Statement**

We aim to predict customer churn (Yes/No) to proactively identify customers at risk of leaving. The business objective is to minimize revenue loss by maximizing recall for churners while maintaining reasonable precision. Given the high cost of losing a customer relative to retention outreach, recall is prioritized over accuracy.

**Metric Selection**

Due to class imbalance and the higher cost of missing churners, Recall is prioritized. Precision-Recall AUC (PR-AUC) is used as a primary model quality metric, as it better reflects churner identification performance than ROC-AUC in imbalanced datasets.

In [10]:
print(f"Dataset Shape: {df.shape}")
print(f"Number of Customers: {df.shape[0]:,}")
print(f"Number of Features: {df.shape[1]}")

Dataset Shape: (7043, 20)
Number of Customers: 7,043
Number of Features: 20


In [11]:
# Dataset information
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [12]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
# Statistical summary
df[["tenure", "MonthlyCharges", "TotalCharges"]].describe()

,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7032.000000
mean,32.371149,64.761692,2283.300441
std,24.559481,30.090047,2266.771362
min,0.000000,18.250000,18.800000
25%,9.000000,35.500000,401.450000
50%,29.000000,70.350000,1397.475000
75%,55.000000,89.850000,3794.737500
max,72.000000,118.750000,8684.800000


### Customer Lifetime Assumption

We assume an average customer lifetime of **36 months (3 years)**.

This assumption is consistent with industry benchmarks for consumer telecommunications services, where typical customer lifetimes range from **2 to 5 years**, depending on market maturity, customer segment, and pricing dynamics. A 36-month lifetime is considered realistic and slightly conservative for consumer services.

This assumption is used for customer lifetime value (LTV) and unit economics calculations throughout the Analysis and Model training.
